# Drawing inductors in Klayout

This Notebook draws inductors in `klayout` instead of `gdspy`.

In [ ]:
import os

import klayout.db as db
import numpy as np

if "sg13g2" not in db.Technology.technology_names():
    tech = db.Technology()
    tech.load("../lib/ihp_open_pdk/ihp-sg13g2/libs.tech/klayout/tech/sg13g2.lyt")
    db.Technology.register_technology(tech)
else: 
    tech = db.Technology.technology_by_name("sg13g2")

In [ ]:

layout = db.Layout()
layout.technology_name = "sg13g2"
top = layout.create_cell("TOP")

inductor = layout.create_cell("Inductor")

# Layer stackup
signal_layer = layout.layer(134, 0)     # TopMetal2
underpass_layer = layout.layer(126, 0)  # TopMetal1

signal_pin_layer = layout.layer(layout.get_info(signal_layer).layer, 2)
signal_underpass_via_layer = layout.layer(layout.get_info(signal_layer).layer - 1, 0)

ind_layer = layout.layer(27,0)
ind_pin_layer = layout.layer(27,2)
ind_text_layer = layout.layer(27,25)

no_fill_layers = [
    layout.layer(1, 23),   # Activ.nofill
    layout.layer(5, 23),   # GatPoly.nofill
    layout.layer(8, 23),   # Metal1.nofill
    layout.layer(10, 23),  # Metal2.nofill
    layout.layer(30, 23),  # Metal3.nofill
    layout.layer(50, 23),  # Metal4.nofill
    layout.layer(67, 23),  # Metal5.nofill
    layout.layer(126, 23), # TopMetal1.nofill
    layout.layer(134, 23), # TopMetal2.nofill
]

no_rcx_layer = layout.layer(148, 0)

print(tech.default_grid())

0.005


In [ ]:
def roundToGrid(tech, num, floor=False, ceil=False):
    """
    Round the number to technology grid.
    If the technology is not defined, or the grid is 0, prints the warning message and returns the same number.
    """
    if tech==None:
        print("WARNING: roundToGrid called, but techonology is not defined")
        return num
    grid = tech.default_grid()
    if grid==0.0:
        print("WARNING: roundToGrid called, but techonology grid=0.0")
        return num
    #print(f"Rounding {num} to grid of size {grid}")
    if floor or ceil:
        return np.floor(num/grid)*grid if floor else np.ceil(num/grid)*grid
    else:
        return np.round(num/grid)*grid

def roundPointToGrid(tech, x, y):
    return roundToGrid(tech, x), roundToGrid(tech, y)

In [ ]:
from inductor_lab.gp.spiral import Topology, design_inductor
from inductor_lab.pdk.sg13g2 import SG13G2

sol = design_inductor(10, 2.4e9, SG13G2, topology=Topology.DIFFERENTIAL)

w = roundToGrid(tech, sol.w * 1e6)             # track width [um]
s = roundToGrid(tech, sol.s * 1e6)             # track spacing [um]
r = roundToGrid(tech, sol.d_out / 2 * 1e6)     # outer radius [um]
N = round(sol.n)                                # number of turns (integer)

conn_len = 3 * w

via_width   = SG13G2.top_vias[0].width    * 1e6   # [um]
via_spacing = SG13G2.top_vias[0].space    * 1e6   # [um]
via_enc     = SG13G2.top_vias[0].enc_upper * 1e6  # [um]
via_pitch   = via_width + via_spacing

e = roundToGrid(tech, w + s + (w + 2*s) / (1 + np.sqrt(2)))

print(f"GP: n={sol.n:.2f} -> N={N}, w={w:.4f} um, s={s:.4f} um, d_out={2*r:.1f} um")
print(f"    L={sol.L*1e9:.3f} nH, Q={sol.Q:.2f}, f_sr={sol.f_sr/1e9:.2f} GHz")

Using solver 'cvxopt'
 for 7 free variables
  in 11 posynomial inequalities.
Solving took 0.107 seconds.
GP: n=7.52 -> N=8, w=8.7950 um, s=2.0000 um, d_out=250.0 um
    L=10.000 nH, Q=10.99, f_sr=13.90 GHz


In [ ]:
def make45Bridge(w, l, s, addVias=False): 
    """
    w : float
        metal width
    l : float
        bridge length
    s : float
        metal separation
    """

    # Start at the origin
    x, y = -(2*w+s)/2, -l/2
    p1 = db.DPoint(*roundPointToGrid(tech, x, y))
    # Move to the corner of the first trace
    x += w
    p2 = db.DPoint(*roundPointToGrid(tech, x, y))
    # Extend this side of the first trace to match DRC
    y += s/(1 + np.sqrt(2))
    p3 = db.DPoint(*roundPointToGrid(tech,x, y))
    # 45 degree turn
    x += w + s
    y += w + s
    p4 = db.DPoint(x, y)
    # Reach the second trace
    y += (w + s)/(1 + np.sqrt(2))
    p5 = db.DPoint(*roundPointToGrid(tech,x, y))
    # Move to the corner of the second trace
    x -= w
    p6 = db.DPoint(*roundPointToGrid(tech,x, y))
    # Extend this side of the second trace to maintain width at w
    y -= s/(1 + np.sqrt(2))
    p7 = db.DPoint(*roundPointToGrid(tech,x, y))
    # Go back to origin
    x -= w + s
    y -= w + s
    p8 = db.DPoint(*roundPointToGrid(tech,x, y))

    # Add connection pads for vias. Connection pads are square shaped and have a side equal to w
    if addVias:
        p1.y -= w
        p2.y -= w
        p5.y += w
        p6.y += w

    ## TODO: Implement fillVias function
    polygon = db.DPolygon([p1, p2, p3, p4, p5, p6, p7, p8])
    return polygon

In [ ]:
def octFill(r) :
    a = r/(1 + np.sqrt(2))  # Half a side of the octagon
    p1 = db.DPoint(-a,-r)
    p2 = db.DPoint(a,-r)
    p3 = db.DPoint(r,-a)
    p4 = db.DPoint(r,a)
    p5 = db.DPoint(a, r)
    p6 = db.DPoint(-a, r)
    p7 = db.DPoint(-r,a)
    p8 = db.DPoint(-r,-a)

    polygon = db.DPolygon([p1, p2, p3, p4, p5, p6, p7, p8])
    return polygon


def octSegment(w, r, e):
    """
    w : float
        metal width
    r : float
        outer radius
    e : float
        spacing for bridges
    """
    ## NOTE: `a` is `side/sqrt(2)`
    a=roundToGrid(tech, (2*r)/(np.sqrt(2)+2))               #  length of outer corner
    r=roundToGrid(tech, r)
    # Inner corner length must be floored instead of rounded
    c=roundToGrid(tech, (2*r)/(np.sqrt(2)+2)+w/(np.sqrt(2)+1), floor=True) #  length of inner corner

    # Start from the outside and leave space for bridges
    x = r
    y = e/2
    p1 = db.DPoint(x,y)
    # Move upwards (vertical segment)
    y = r - a
    p2 = db.DPoint(x,y)
    # 45 degree turn
    x -= a
    y += a
    p3 = db.DPoint(x,y)
    # Get to the end of the horizontal segment
    x = e/2
    p4 = db.DPoint(x,y)
    # Move downwards to begin the inner trace
    y -= w
    p5 = db.DPoint(x,y)
    # Complete the horizontal segment
    x = r - c
    p6 = db.DPoint(x,y)
    # 45 degree turn
    x = r - w
    y = r - c
    p7 = db.DPoint(x,y)
    # Complete the vertical segment
    y = e/2
    p8 = db.DPoint(x,y)

    polygon = db.DPolygon([p1, p2, p3, p4, p5, p6, p7, p8])
    return polygon


In [ ]:
# Draw the segments
for n in range(N):
    seg = octSegment(w, r - n *(w + s), e)
    for i in range(4):
        seg.transform(db.DTrans(rot=45*i))
        inductor.shapes(signal_layer).insert(seg)

# Draw underpasses and bridges
via_cell = layout.create_cell("VIA")
via_cell.shapes(signal_underpass_via_layer).insert(db.DBox(0,0,via_width,via_width))
for n in range(N-1):
    bridge = make45Bridge(w, e, s, addVias=True)
    bridge.transform(db.DTrans(rot=45))
    bridge.transform(db.DTrans( 0, (-1)**(2 + n)*(r - (1+n)*(w + s) + s/2) ))
    upass = bridge.transformed(db.DTrans(rot=90, mirrx=True))
    inductor.shapes(signal_layer).insert(bridge)
    inductor.shapes(underpass_layer).insert(upass)
    # Add vias to the underpass
    left_bottom = min(upass.each_point_hull(), key=lambda p: (p.x, p.y))
    right_bottom = max(upass.each_point_hull(), key=lambda p: (p.x, -p.y))
    via_number = np.floor((w - 2*via_enc + via_spacing) / via_pitch)
    array_size = via_number * via_width + (via_number - 1) * via_spacing
    centering_offset = via_enc + (w - 2*via_enc - array_size)/2
    left_vias = db.DCellInstArray(
        via_cell.cell_index(),
        db.DTrans(left_bottom.x  + centering_offset, left_bottom.y + centering_offset),
        db.DVector(via_pitch,0),
        db.DVector(0,via_pitch),
        via_number,
        via_number
    )
    right_vias = db.DCellInstArray(
        via_cell.cell_index(),
        db.DTrans(right_bottom.x  - centering_offset - via_width, right_bottom.y + centering_offset),
        db.DVector(-via_pitch,0),
        db.DVector(0,via_pitch),
        via_number,
        via_number
    )

    inductor.insert(left_vias)
    inductor.insert(right_vias)


# Patch segments to form turns
for n in range(N):
    patch = db.DBox(db.DPoint(r, e/2), db.DPoint(r-w, -e/2))
    patch.move(-n*(w + s), 0)
    inductor.shapes(signal_layer).insert(patch)
    inductor.shapes(signal_layer).insert(patch.transformed(db.DTrans(rot=90)))

# Patch the inner turn
patch = db.DBox(db.DPoint(e/2, r - (N-1)*(w + s)), db.DPoint(-e/2, r-w- (N-1)*(w + s)))
if not (N % 2):
    patch = patch.transformed(db.DTrans(rot=90))
inductor.shapes(signal_layer).insert(patch)

# Draw the connections
conn = db.DBox(db.DPoint(-e/2 - w/2, -r + w), db.DPoint(-e/2 + w/2, -r - conn_len))
inductor.shapes(signal_layer).insert(conn)
inductor.shapes(signal_layer).insert(conn.moved(e, 0))

# Merge all polygons in signal layer
region = db.Region(inductor.begin_shapes_rec(signal_layer))
region.merge()
inductor.shapes(signal_layer).clear()
inductor.shapes(signal_layer).insert(region)

# Draw the pins
pin_height = s
pin = db.DBox(db.DPoint(-e/2 - w/2, -r - conn_len + pin_height), db.DPoint(-e/2 + w/2, -r - conn_len))
inductor.shapes(signal_pin_layer).insert(pin)
inductor.shapes(signal_pin_layer).insert(pin.moved(e, 0))
inductor.shapes(ind_pin_layer).insert(pin)
inductor.shapes(ind_pin_layer).insert(pin.moved(e, 0))

# Draw "No fill" regions
oct = octFill(r + conn_len)
for layer in no_fill_layers:
    inductor.shapes(layer).insert(oct)

# Prevent parasitics extraction
inductor.shapes(no_rcx_layer).insert(oct)

# Mark the region as an inductor
inductor.shapes(ind_layer).insert(oct)

# Label pins
pin1_label = db.DText("P1", -e/2 - w/2, -r - conn_len)
pin2_label = db.DText("P2", e/2 - w/2, -r - conn_len)
inductor.shapes(ind_text_layer).insert(pin1_label)
inductor.shapes(ind_text_layer).insert(pin2_label)



text ('P2',r0 3650,-151385)

In [ ]:
# Put the inductor cell as a child inside TOP
ind_cell = db.CellInstArray(inductor.cell_index(), db.Trans())
top.insert(ind_cell)

ind_name = f"inductor_{w}_{s}_{2*r}_{N}"
out_file = ".out/" + ind_name + ".gds"
os.makedirs(".out/", exist_ok=True)
layout.write(out_file)

In [ ]:
# Create a gds2palace compatible file also
top.clear()

# Place each pin in a different layer
palace_pin1_layer = layout.layer(201, 0)
palace_pin2_layer = layout.layer(202, 0)

inductor.shapes(palace_pin1_layer).insert(pin.enlarged(0, -pin.height()/4).moved(0, -pin.height()/4))
inductor.shapes(palace_pin2_layer).insert(pin.enlarged(0, -pin.height()/4).moved(e, -pin.height()/4))

# Add a ground frame
palace_ground_layer = layout.layer(8, 0)  # Metal1

ind_bbox = inductor.dbbox()

gframe = db.DPolygon(ind_bbox.enlarged(roundToGrid(tech, r/2 + 5*w)))
gframe_hole = ind_bbox.enlarged(roundToGrid(tech, r/2))
gframe.insert_hole(gframe_hole)

gframe_pin = db.DBox(db.DPoint(pin.left - 5, pin.top),  db.DPoint(pin.right + e + 5, gframe_hole.bottom))

inductor.shapes(palace_ground_layer).insert(gframe)
inductor.shapes(palace_ground_layer).insert(gframe_pin)

# Merge all polygons in groundframe layer
region = db.Region(inductor.begin_shapes_rec(palace_ground_layer))
region.merge()
inductor.shapes(palace_ground_layer).clear()
inductor.shapes(palace_ground_layer).insert(region)

ind_cell = db.CellInstArray(inductor.cell_index(), db.Trans())
top.insert(ind_cell)
em_gds = ".out/"+ ind_name + "_forEM.gds"
layout.write(em_gds)


# `gds2palace` flow

In [ ]:
import gds2palace as gp
import os, subprocess, pathlib, sys

In [ ]:
## Add the scripts folder to path
notebook_root = pathlib.Path(os.getcwd())
repo_root     = notebook_root.parent.absolute()
scripts_root  = repo_root / pathlib.Path("scripts") 

print(str(scripts_root))
os.environ["PATH"] += os.pathsep + os.pathsep.join([str(scripts_root)])
print(os.environ["PATH"])

#os.environ["DISPLAY"] = "localhost:12.0"

/home/try_except/Media/DualStorage/GitRepos/InductorLab/scripts
/home/try_except/.pyenv/versions/InductorLab/bin:/home/try_except/.pyenv/shims:/home/try_except/.pyenv/bin:/home/try_except/.local/bin:/home/try_except/.cargo/bin:/home/try_except/Tools/platform-tools:/home/try_except/.pyenv/bin:/usr/local/bin:/usr/bin:/bin:/usr/local/games:/usr/games:/home/try_except/Media/DualStorage/GitRepos/InductorLab/scripts


In [ ]:
em_gds = ".out/" + ind_name + '_forEM' + '.gds'

In [ ]:
# ======================== workflow settings ================================

#start solver after creating the model?
start_simulation = True
run_command = ['./run_sim']

In [ ]:
# ===================== input files and path settings =======================

gds_filename = em_gds
XML_filename = "../resources/SG13G2_200um.xml"

preprocess_gds = True

sim_path = gp.utilities.create_sim_path(".out/", ind_name)
print('Simulation data directory: ', sim_path)

In [ ]:
# ======================== simulation settings ================================

settings = {}

settings['unit']   = 1e-6
settings['margin'] = 50

settings['fpoint'] = [2.4e9]           # solve only at operating frequency
settings['adaptive_sweep'] = False     # no sweep to adapt over

settings['refined_cellsize']         = 1
settings['cells_per_wavelength']     = 10
settings['meshsize_max']             = 70
settings['adaptive_mesh_iterations'] = 0

settings['order']      = 2
settings['no_gui']     = True
settings['no_preview'] = True

settings['merge_polygon_size'] = 1.5


# Ports
simulation_ports = gp.simulation_setup.all_simulation_ports()
simulation_ports.add_port(gp.simulation_setup.simulation_port(portnumber=1, voltage=1, port_Z0=50, source_layernum=201, from_layername='Metal1', to_layername='TopMetal2', direction='z'))
simulation_ports.add_port(gp.simulation_setup.simulation_port(portnumber=2, voltage=1, port_Z0=50, source_layernum=202, from_layername='Metal1', to_layername='TopMetal2', direction='z'))

In [ ]:
# ======================== meshing ================================

materials_list, dielectrics_list, metals_list = gp.stackup_reader.read_substrate(XML_filename)
layernumbers = metals_list.getlayernumbers()
layernumbers.extend(simulation_ports.portlayers)

allpolygons = gp.gds_reader.read_gds(
    gds_filename, layernumbers, purposelist=[0],
    metals_list=metals_list,
    preprocess=preprocess_gds,
    merge_polygon_size=settings['merge_polygon_size'],
    cellname="Inductor",
)

settings['simulation_ports'] = simulation_ports
settings['materials_list']   = materials_list
settings['dielectrics_list'] = dielectrics_list
settings['metals_list']      = metals_list
settings['layernumbers']     = layernumbers
settings['allpolygons']      = allpolygons
settings['sim_path']         = sim_path
settings['model_basename']   = ind_name

excite_ports = simulation_ports.all_active_excitations()
config_name, data_dir = gp.simulation_setup.create_palace(excite_ports, settings)
gp.utilities.create_run_script(sim_path)

print(f"Model written to {sim_path}")

In [ ]:
# ======================== simulation ================================

if start_simulation:
    run_palace  = str(scripts_root / 'run_palace')
    combine_snp = str(scripts_root / 'combine_snp')

    os.chdir(sim_path)
    try:
        subprocess.run([run_palace, 'config.json'], check=True)
        subprocess.run([combine_snp], check=True)
    except subprocess.CalledProcessError as e:
        print(f"Step failed with return code {e.returncode}")
    except FileNotFoundError as e:
        print(f"Command not found: {e}")
    finally:
        os.chdir(notebook_root)

>> /usr/lib64/mpich/bin/mpirun -n 4 /opt/palace/bin/palace-x86_64.bin config.json

_____________     _______
_____   __   \____ __   /____ ____________
____   /_/  /  __ ` /  /  __ ` /  ___/  _ \
___   _____/  /_/  /  /  /_/  /  /__/  ___/
  /__/     \___,__/__/\___,__/\_____\_____/


--> Warning!
Output folder is not empty; program will overwrite content! (output/inductor_8.795_2.0_250.0_8)
Git changeset ID: v0.16.0-34-gea2e7b23
Running with 4 MPI processes, 1 OpenMP thread
Device configuration: omp,cpu
Memory configuration: host-std
libCEED backend: /cpu/self/xsmm/blocked

Added 17942 boundary elements for material interfaces to the mesh
Finished partitioning mesh into 4 subdomains

Characteristic length and time scales:
 Lc = 9.157e-04 m, tc = 3.055e-03 ns

Mesh curvature order: 1
Mesh bounding box:
 (Xmin, Ymin, Zmin) = (-4.579e-04, -4.579e-04, -5.000e-05) m
 (Xmax, Ymax, Zmax) = (+4.579e-04, +4.579e-04, +4.499e-04) m

Parallel Mesh Stats:

                minimum     average     m

KeyboardInterrupt: 

In [ ]:
# ======================== combine S-parameters (recovery cell) ================================
# Run this cell if combine_snp did not execute after the simulation.

combine_snp = str(scripts_root / 'combine_snp')

os.chdir(sim_path)
try:
    subprocess.run([combine_snp], check=True)
except subprocess.CalledProcessError as e:
    print(f"combine_snp failed with return code {e.returncode}")
finally:
    os.chdir(notebook_root)

Found extra file with port information: /home/try_except/Media/DualStorage/GitRepos/InductorLab/notebooks/.out/palace_model/inductor_8.795_2.0_250.0_8_data/port_information.json
Port Z0 values found: [50, 50]
Port impedance for Touchstone header:  50
Number of ports:  2
Created combined S-parameter file for  2 ports, filename:  /home/try_except/Media/DualStorage/GitRepos/InductorLab/notebooks/.out/palace_model/inductor_8.795_2.0_250.0_8_data/output/inductor_8.795_2.0_250.0_8/inductor_8.795_2.0_250.0_8.s2p
Created file with DC extrapolation:  /home/try_except/Media/DualStorage/GitRepos/InductorLab/notebooks/.out/palace_model/inductor_8.795_2.0_250.0_8_data/output/inductor_8.795_2.0_250.0_8/inductor_8.795_2.0_250.0_8_dc 

Port de-embedding based on port geometry data
Cascading L= -2.93 pH at port 1
Cascading L= -2.93 pH at port 2
Created file with de-embedding (cascaded negative port L):  /home/try_except/Media/DualStorage/GitRepos/InductorLab/notebooks/.out/palace_model/inductor_8.795_2

# Simulation results

In [ ]:
import skrf as rf
from skrf.util import find_nearest_index

freq  = 2.4e9
omega = 2 * np.pi * freq

snp_path = os.path.join(sim_path, 'output', ind_name, f"{ind_name}_deembedded")
network = rf.Network(snp_path)

z11 = network.z[0::, 0, 0]
z21 = network.z[0::, 1, 0]
z12 = network.z[0::, 0, 1]
z22 = network.z[0::, 1, 1]

Zdiff = z11 - z12 - z21 + z22
Ldiff = Zdiff.imag / omega
Rdiff = Zdiff.real
Qdiff = Zdiff.imag / Zdiff.real

findex = find_nearest_index(network.frequency.f, freq)
print(f"""Zdiff = {Zdiff[findex]}
Ldiff = {Ldiff[findex] * 1e9:.3f} nH
Rdiff = {Rdiff[findex]:.3f} ohm
Qdiff = {Qdiff[findex]:.3f}
""")